In [ ]:
using DifferentialEquations
using Plots
using LinearAlgebra

In [ ]:
f= (u, p, t) -> u^2 -t

In [ ]:

u0 = 1.0
tspan = (0.0, 1.0)

prob = ODEProblem(f, u0, tspan)
sol = solve(prob)

In [ ]:
sol.u

In [ ]:
plot(sol)

In [ ]:
function advection1!(du, u, p, t)
    c = p[1];
    Δx = p[2];
    u_left = p[3];
    du[1] = -c * (u[1] - u_left) / Δx;
    for i in 2:length(u)
        du[i] = -c * (u[i] - u[i-1]) / Δx;
    end
    du
end

In [ ]:
xx = LinRange(-5.0, 5.0, 51)[2:end];

u0 = exp.(-xx.^2)
tspan = (0.0, 5.0)

c = 3.0;
u_left = 0.0;
@show Δx = xx[2] - xx[1];
p = [c, Δx, u_left];

dt = 1e-1

@show c * dt / Δx;

prob = ODEProblem(advection1!, u0, tspan, p)
sol = solve(prob, Euler(); dt=dt, adaptive=false, saveat=.1)

In [ ]:
plot(xx, sol.u[1], label="t=$(sol.t[1])")
plot!(xx, sol.u[2], label="t=$(sol.t[2])")
plot!(xx, sol.u[10], label="t=$(sol.t[10])")
plot!(xx, sol.u[end], label="t=$(sol.t[end])")

In [ ]:
anim = @animate for i in 1:length(sol.t)
    plot(xx, sol.u[i], label="t=$(sol.t[i])", ylim=(0.0, 1.0))
end

In [ ]:
gif(anim, "advection.gif", fps=60)

In [ ]:
function backward_difference_matrix(n; h=1.0)
    D = zeros(n, n)

    for i in 1:n
        D[i, i] = 1/h
        if i > 1
            D[i, i-1] = -1/h
        end
    end

    return D
end


In [ ]:
backward_difference_matrix(5)

In [ ]:
sigma_vals = LinRange(0.0, 2.0, 21);
n = 20;
matnorm_vals = [];
for sigma in sigma_vals
    D = backward_difference_matrix(n, h=1.0)
    A = I - sigma  * D;
    vals = eigvals(A)
    push!(matnorm_vals, opnorm(A, Inf))
end

In [ ]:
plot(sigma_vals, matnorm_vals, label="Spectral Radius", xlabel="Sigma", ylabel="Spectral Radius", title="Spectral Radius vs Sigma")

In [ ]:
function advection2!(du, u, p, t)
    c = p[1];
    Δx = p[2];
    phi = p[3];
    f = p[4];
    x = p[5];


    du[1] = -c * (u[1] - phi(t)) / Δx + f(x[1], t);
    for i in 2:length(u)
        du[i] = -c * (u[i] - u[i-1]) / Δx + f(x[i], t);
    end
    du
end

In [ ]:
xx = LinRange(-5.0, 5.0, 201)[2:end];

# u0 = exp.(-xx.^2)
u0 = zeros(length(xx))
tspan = (0.0, 10.0)

c = 3.0;
phi(t) = sin(0.1 * π * t)^2;
# source(x,t) = 0.1 * cos(π * x * t);
source(x,t) = 0.0;
@show Δx = xx[2] - xx[1];

p = [c, Δx, phi, source, xx];

dt = 1e-4

@show c * dt / Δx;

prob = ODEProblem(advection2!, u0, tspan, p)
# sol = solve(prob, Euler(); dt=dt, adaptive=false, saveat=.1);
sol = solve(prob, saveat=.1);

In [ ]:
anim = @animate for i in 1:length(sol.t)
    plot(xx, sol.u[i], label="t=$(sol.t[i])", ylim=(0.0, 1.0))
    plot!([-5],[phi(sol.t[i])], seriestype=:scatter, label="phi(t)")
end

In [ ]:
gif(anim, "advection.gif", fps=60)

In [ ]:
function heat1!(du, u, p, t)
    α = p[1];
    δx = p[2];


    du[1] = α/ δx^2 * (u[2] -2* u[1]) 
    for i in 2:length(u)-1
        du[i] = α/ δx^2 * (u[i+1] -2* u[i] + u[i-1]);
    end
    du[end] = α/ δx^2 * (-2* u[end] + u[end-1] );
    du
end

In [ ]:
xx = LinRange(-5.0, 5.0, 2001)[2:end-1];

u0 = ones(length(xx))
# u0 = zeros(length(xx))
tspan = (0.0, 10.0)

alpha = 1.0;
@show Δx = xx[2] - xx[1];

p = [alpha, Δx];

# dt = 1e-4

# @show alpha * dt / Δx;

prob = ODEProblem(heat1!, u0, tspan, p)
# sol = solve(prob, Euler(); dt=dt, adaptive=false, saveat=.1);
sol = solve(prob, saveat=.1);

In [ ]:
sol.alg

In [ ]:
anim = @animate for i in 1:length(sol.t)
    plot(xx, sol.u[i], label="t=$(sol.t[i])", ylim=(0.0, 1.0))
end

In [ ]:
gif(anim, "heat.gif", fps=60)

In [ ]:
xx = LinRange(-5.0, 5.0, 401)[2:end-1];

u0 = ones(length(xx))
# u0 = zeros(length(xx))
tspan = (0.0, 10.0)

alpha = 1.0;
@show Δx = xx[2] - xx[1];

p = [alpha, Δx];

dt = 1e-3

@show alpha * dt / Δx^2;

prob = ODEProblem(heat1!, u0, tspan, p)
sol = solve(prob, Euler(); dt=dt, adaptive=false, saveat=.1);
# sol = solve(prob, saveat=.1);
anim = @animate for i in 1:length(sol.t)
    plot(xx, sol.u[i], label="t=$(sol.t[i])", ylim=(0.0, 1.0))
end

gif(anim, "heat.gif", fps=60)

In [ ]:
xx = LinRange(-5.0, 5.0, 401)[2:end-1];

u0 = ones(length(xx))
# u0 = zeros(length(xx))
tspan = (0.0, 10.0)

alpha = 1.0;
@show Δx = xx[2] - xx[1];

p = [alpha, Δx];

dt = 1e-3

@show alpha * dt / Δx^2;

prob = ODEProblem(heat1!, u0, tspan, p)
sol = solve(prob, ImplicitEuler(); dt=dt, adaptive=false, saveat=.1);
# sol = solve(prob, saveat=.1);
anim = @animate for i in 1:length(sol.t)
    plot(xx, sol.u[i], label="t=$(sol.t[i])", ylim=(0.0, 1.0))
end

gif(anim, "heat.gif", fps=60)